In [1]:
import marimo as mo

# knot — 02: ingest

Bind a source to a class. See the write SQL knot emits. Bind row
data to it via the connector. Verify what landed.

In [2]:
import json
import uuid

import psycopg

In [3]:
# Same shared spec as 01_deploy — imports the spec, the Movie class,
# the imdb Source, and the imdb→Movie binding from ``movies_spec.py``.
# Real deployments share this exact import pattern: workers, the
# service API, the ER pipeline all import from the same spec module.
from movies_spec import imdb, imdb_movie_b, movie, spec

imdb_movie_b

SourceBinding(source=Source(name='imdb', description=None), class_=OntologyClass(name='Movie', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='title', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='year', type=<Primitive.INTEGER: 'integer'>, identifier=False, required=False, description=None), Slot(name='director', type=ClassRef(target=OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='name', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='birth_country', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None)], description=None)), identifier=False, required=False, desc

In [4]:
# Host plumbing + deploy. Schema name is a throwaway per-run id.
pg = psycopg.connect(
    host="localhost", port=5433,
    user="knot", password="knot", dbname="knot",
    autocommit=True,
)
schema = f"knot_play_{uuid.uuid4().hex[:8]}"
pg.execute(spec.init_sql(schema=schema))
schema

'knot_play_cc0cf2ec'

In [5]:
# Load real sample data from disk — imdb's movies.json. Each row
# carries:
#   * `source_identifier` — imdb's own key (e.g. "tt1838941").
#     This is the only stable identity imdb knows about; knot's
#     cross-source `canonical_id` doesn't exist yet — ER assigns
#     it later (see 03_er).
#   * the class slots (`title`, `year`, `director`) as native values.
#   * extras (`imdb_rating`, `num_votes`, `box_office_usd`, …) that
#     aren't in the spec — they ride along in the row dict and land
#     in `raw_payload jsonb` on the binding row, recoverable later
#     without re-fetching from imdb.
#
# In a real worker this loader would be a pandas DataFrame from a
# CSV/parquet, a Kafka pull, an HTTP fetch — anything that yields a
# list[dict]. knot only cares about the dict shape.
from pathlib import Path

DATA = Path("../data/movies/imdb/movies.json")
rows = json.loads(DATA.read_text())
print(f"loaded {len(rows)} rows; first one:")
rows[0]

loaded 70 rows; first one:


{'source_identifier': 'tt1838941',
 'title': 'Reservoir Dogs',
 'year': 1992,
 'runtime_minutes': 98,
 'director': 'p_tarantino',
 'imdb_rating': 7.8,
 'num_votes': 1651568,
 'mpaa_rating': 'G',
 'aspect_ratio': '1.43 : 1 (IMAX)',
 'color_info': 'Color',
 'box_office_usd': 1384664366,
 'country_of_origin': ['USA']}

In [6]:
# `binding.write_sql()` returns two SQL templates — both reference a
# single `%(rows)s::jsonb` parameter. knot never touches the rows;
# the host's connector binds them at execute time.
close_out, insert = imdb_movie_b.write_sql(schema=schema)
print("--- close_out ---")
print(close_out)
print("\n--- insert ---")
print(insert)

--- close_out ---
UPDATE knot_play_cc0cf2ec.movie_bindings AS b
SET valid_to = now()
FROM (
    SELECT
        (r->>'canonical_id') AS canonical_id,
        (r->>'source_identifier') AS source_identifier
    FROM jsonb_array_elements(%(rows)s::jsonb) AS r
) AS keys
WHERE b.canonical_id = keys.canonical_id
  AND b.source_name = 'imdb'
  AND b.source_identifier = keys.source_identifier
  AND b.valid_to IS NULL;

--- insert ---
INSERT INTO knot_play_cc0cf2ec.movie_bindings (source_name, source_identifier, canonical_id, title, year, director, title_embedding, raw_payload)
SELECT
    'imdb',
    raw.source_identifier,
    raw.canonical_id::text,
    raw.title::text,
    raw.year::integer,
    raw.director::text,
    raw.title_embedding::vector(384),
    raw.__raw_payload
FROM (
    SELECT
        r AS __raw_payload,
        (r->>'source_identifier') AS source_identifier,
        (r->>'canonical_id') AS canonical_id,
        (r->>'title') AS title,
        (r->>'year') AS year,
        (r->>

In [7]:
# Run both statements with the rows bound as a single jsonb param.
# For an autocommit connection each cur.execute commits independently;
# wrap in pg.transaction() if you want atomic close_out + insert.
payload = json.dumps(rows)
with pg.cursor() as _cur:
    _cur.execute(close_out, {'rows': payload})
    _cur.execute(insert, {'rows': payload})

In [8]:
# Verify via ``movie.from_source(imdb)`` — one source's claims about
# Movie. This is a Query over the raw bindings layer (one row per
# source_identifier) scoped to ``source_name = 'imdb'``. The
# ``resolved`` layer would be empty here: ER hasn't run yet, so
# every row's ``canonical_id`` is still NULL.
q = (
    movie.from_source(imdb)
    .order_by(movie.col.year, "desc")
    .limit(10)
    .select(movie.col.canonical_id, movie.col.title, movie.col.year)
)
sql = q.sql(schema=schema)
with pg.cursor() as _cur:
    _cur.execute(sql)
    cols = [d.name for d in _cur.description]
    for row in _cur.fetchall():
        print(dict(zip(cols, row)))

{'canonical_id': None, 'title': 'Dune: Part Two', 'year': 2024}
{'canonical_id': None, 'title': 'Oppenheimer', 'year': 2023}
{'canonical_id': None, 'title': 'Poor Things', 'year': 2023}
{'canonical_id': None, 'title': 'Barbie', 'year': 2023}
{'canonical_id': None, 'title': 'The Northman', 'year': 2022}
{'canonical_id': None, 'title': 'Dune', 'year': 2021}
{'canonical_id': None, 'title': 'Midsommar', 'year': 2019}
{'canonical_id': None, 'title': 'Little Women', 'year': 2019}
{'canonical_id': None, 'title': 'The Lighthouse', 'year': 2019}
{'canonical_id': None, 'title': 'Pain and Glory', 'year': 2019}
